# PCA biplots: surface N2O flux and MLS N2O/O3 at the flux sites

A cleaned version of `pca_test3.ipynb`. It produces two biplots from the same data:

- **A. Original method.** Standardise the ten variables and run PCA, as in your third biplot cell.
- **B. Transformed fluxes.** Same, except the surface flux columns are passed through a signed log transform before standardising.

A comparison section at the end puts the two side by side: explained variance, loadings, and a scree plot.

Rows sharing a location and month are averaged into one point before anything else, as before.

## 1. Settings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import adjustText
from adjustText import adjust_text
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from matplotlib.patches import Ellipse
from scipy.stats import chi2
from scipy.stats import f as f_dist

INPUT_FILE = "data_pca4.csv"      # written by the merged prep notebook (sep=";")
DATE_TAG   = "202610"             # appended to output file names

FEATURES = ["N2O_wetland", "N2O_inventory",
            "N2O_68hPa", "N2O_46hPa", "N2O_32hPa", "N2O_22hPa",
            "O3_68hPa", "O3_46hPa", "O3_32hPa", "O3_22hPa"]

# Columns that get the signed log transform in variant B
TRANSFORM_COLS = ["N2O_wetland", "N2O_inventory"]

# False = one label per location, on its first point (your previous method).
# True  = label every point; locations with several months get the month added.
LABEL_ALL_POINTS = False

ARROW_SCALE = 3                   # loading arrows are drawn x3, as before
GROUP_COLORS = {1: "steelblue", 2: "darkorange", 3: "forestgreen"}
GROUP_NAMES  = {1: "NH", 2: "Tropics", 3: "SH"}
# Ellipses: which latitude groups share one ellipse. Point colours still follow GROUP_COLORS.
# Use {} for no ellipses, or e.g. {"NH": [1], "Tropics": [2], "SH": [3]} for separate ones.
ELLIPSE_GROUPS = {"Extratropics": [1, 3], "Tropics": [2]}
ELLIPSE_COLORS = {"Extratropics": "slategray", "Tropics": "darkorange"}
ELLIPSE_KIND = "data"   # "data" = spread of the points; "mean" = confidence region for the group mean
ELLIPSE_LEVEL = 0.865        # data ellipse: ~95% of a group's points if roughly normal; 0.393 for one standard deviation, 0.865 for two
ELLIPSE_MIN_POINTS = 6      # groups with fewer points get no ellipse (too few for a covariance)

## 2. Load and merge repeated measurements
Several flux measurements share a location and month, so they share identical MLS values. They are averaged into one row per location-month. The flux is averaged, and the MLS columns stay as they are because they're already identical within a group. The latitude group is computed afterwards from `lat`, using the same thresholds as before.

In [ ]:
raw = pd.read_csv(INPUT_FILE, sep=";")

def assign_group(lat):
    """1 = NH (lat > 30), 3 = SH (lat < -30), 2 = tropics."""
    return 1 if lat > 30 else 3 if lat < -30 else 2

agg = {col: "mean" for col in FEATURES} | {col: "first" for col in ["lat", "lon", "loc_inname"]}
df = raw.groupby(["Asukoht", "year", "month"], as_index=False).agg(agg)
df["Group"] = df["lat"].apply(assign_group)

print(f"{len(raw)} measurements -> {len(df)} location-months at {df['Asukoht'].nunique()} locations")

## 3. Feature tables for the two variants
The flux is strongly skewed: most values are near zero, and a few sites (Brunei, Uganda E and S) are in the hundreds. Standardising divides by a standard deviation that those few points dominate, so they largely set the flux arrow. A log transform compresses the high values. Because some fluxes are slightly negative (uptake), the signed version `sign(x) · log(1 + |x|)` is used. It keeps zero at zero and keeps the sign of negative values.

The transform is applied to the location-month means from step 2, just before standardisation. The skewness printout shows how much it changes each transformed column. Zero means symmetric.

In [ ]:
def signed_log1p(x):
    return np.sign(x) * np.log1p(np.abs(x))

X_orig = df[FEATURES].copy()
X_tr   = X_orig.copy()
X_tr[TRANSFORM_COLS] = signed_log1p(X_tr[TRANSFORM_COLS])

pd.DataFrame({"skew before": X_orig[TRANSFORM_COLS].skew(),
              "skew after":  X_tr[TRANSFORM_COLS].skew()}).round(2)

## 4. PCA
Both variants: standardise, then fit a PCA with all components. The first two give the biplot, and all of them feed the scree plot. The result is identical to fitting only two components.

The sign of a principal component is arbitrary. The same pattern can come out mirrored between two runs. So variant B's components are flipped where needed to point the same way as variant A's. This changes nothing about the analysis. It just makes the two biplots directly comparable.

In [ ]:
def run_pca(X):
    """Standardise X and fit a full PCA; return scores, loadings and explained variance (%)."""
    Z = StandardScaler().fit_transform(X)
    pca = PCA().fit(Z)
    return pca.transform(Z), pca.components_.T, pca.explained_variance_ratio_ * 100

scores_A, load_A, var_A = run_pca(X_orig)
scores_B, load_B, var_B = run_pca(X_tr)

for k in range(2):                                   # align PC1/PC2 signs of B with A
    if np.corrcoef(scores_A[:, k], scores_B[:, k])[0, 1] < 0:
        scores_B[:, k] *= -1
        load_B[:, k] *= -1

print(f"A (original):    PC1 {var_A[0]:.1f}%, PC2 {var_A[1]:.1f}%")
print(f"B (transformed): PC1 {var_B[0]:.1f}%, PC2 {var_B[1]:.1f}%")

## 5. Biplot function
This is your third biplot cell turned into a function, so both variants are drawn the same way:
- points coloured by latitude group
- one label per location on its first point, unless `LABEL_ALL_POINTS = True`
- loading arrows ×`ARROW_SCALE`
- arrow labels included in `adjust_text`, so they don't overlap the site labels

`adjust_text` changed its arguments in version 1.0. Your old ones (`expand_points`, `force_points`) are silently ignored by newer versions, so the helper passes whichever set matches the installed version.

In [ ]:
ARROW_LABELS = {"N2O_wetland": "Flux", "N2O_inventory": "EDGAR",
                **{f"N2O_{p}hPa": f"N₂O {p}" for p in (68, 46, 32, 22)},
                **{f"O3_{p}hPa": f"O₃ {p}" for p in (68, 46, 32, 22)}}
# in biplot(), replace `feature` in the arrow-label ax.text(...) with ARROW_LABELS.get(feature, feature)

In [ ]:
def spread_labels(texts):
    if adjustText.__version__.startswith("0."):
        adjust_text(texts, expand_points=(1.2, 1.4), force_text=(0.1, 0.1), force_points=(0.5, 0.5))
    else:
        adjust_text(texts, expand=(1.2, 1.4), force_text=(0.1, 0.1), force_static=(0.5, 0.5))



def data_ellipse(ax, xy, color, label, level=ELLIPSE_LEVEL, kind=ELLIPSE_KIND):
    """Data ellipse (spread of points) or confidence ellipse of the group mean, in PC space."""
    n = len(xy)
    mean, cov = xy.mean(axis=0), np.cov(xy, rowvar=False)
    if kind == "mean":
        # Hotelling's T²: exact for small groups, unlike the chi-square approximation
        r2 = 2 * (n - 1) / (n * (n - 2)) * f_dist.ppf(level, 2, n - 2)
        tag = f"{level:.0%} CI of mean"
    else:
        r2 = chi2.ppf(level, 2)
        tag = f"{np.sqrt(r2):.0f}σ"
    vals, vecs = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(vecs[1, 1], vecs[0, 1]))
    width, height = 2 * np.sqrt(vals[::-1] * r2)
    ax.add_patch(Ellipse(mean, width, height, angle=angle, facecolor=color, edgecolor=color,
                         alpha=0.1, lw=1.5, label=f"{label} ({tag})", zorder=0))

# --- Arrow labels: fixed beside each arrow tip, not moved by adjust_text ---
ARROW_LABELS = {"N2O_wetland": "Flux", "N2O_inventory": "EDGAR",
                **{f"N2O_{p}hPa": f"N₂O {p}" for p in (68, 46, 32, 22)},
                **{f"O3_{p}hPa": f"O₃ {p}" for p in (68, 46, 32, 22)}}
# Arrows that nearly coincide share one label, placed at their mean tip
ARROW_LABEL_GROUPS = {"N₂O 22–46": ["N2O_22hPa", "N2O_32hPa", "N2O_46hPa"]}
ARROW_LABEL_PAD = 0.12      # gap between arrow tip and label, in PC units
ARROW_LABEL_NUDGE = {}      # hand fixes in PC units, e.g. {"O₃ 46": (0.0, 0.15)}; keys are the shown labels
 
 
def spread_labels(texts, x, y):
    """Move site labels apart, also avoiding the points (x, y)."""
    if adjustText.__version__.startswith("0."):
        adjust_text(texts, x=x, y=y, expand_points=(1.3, 1.6), force_text=(0.3, 0.3), force_points=(0.5, 0.5))
    else:
        adjust_text(texts, x=x, y=y, expand=(1.3, 1.6), force_text=(0.3, 0.3), force_static=(0.5, 0.5))
 
 
def place_arrow_label(ax, tip, text):
    """Put a label just beyond an arrow tip, aligned away from the origin."""
    u = tip / np.hypot(*tip)
    x, y = tip + u * ARROW_LABEL_PAD + np.asarray(ARROW_LABEL_NUDGE.get(text, (0, 0)))
    ha = "left" if u[0] > 0.3 else "right" if u[0] < -0.3 else "center"
    va = "bottom" if u[1] > 0.3 else "top" if u[1] < -0.3 else "center"
    return ax.text(x, y, text, color="red", fontsize=10, ha=ha, va=va, zorder=5,
                   bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.75))
 
 
def text_outline_points(fig, ax, t, n=4):
    """Points on and inside a text's bounding box, in data units (obstacles for adjust_text)."""
    bb = t.get_window_extent(renderer=fig.canvas.get_renderer()).transformed(ax.transData.inverted())
    gx, gy = np.meshgrid(np.linspace(bb.x0, bb.x1, n), np.linspace(bb.y0, bb.y1, 2))
    return gx.ravel(), gy.ravel()
 
 
def biplot(scores, loadings, var, fname, title=None):
    pc = pd.DataFrame(scores[:, :2], columns=["PC1", "PC2"], index=df.index)
    fig, ax = plt.subplots(figsize=(10, 8))
 
    for group in sorted(df["Group"].unique()):
        sel = df["Group"] == group
        ax.scatter(pc.loc[sel, "PC1"], pc.loc[sel, "PC2"], s=50, alpha=1,
                   color=GROUP_COLORS[group], label=GROUP_NAMES[group], zorder=3)
 
    for name, groups in ELLIPSE_GROUPS.items():
        xy = pc.loc[df["Group"].isin(groups), ["PC1", "PC2"]].to_numpy()
        if len(xy) < ELLIPSE_MIN_POINTS:
            print(f"No ellipse for {name}: only {len(xy)} points")
            continue
        data_ellipse(ax, xy, ELLIPSE_COLORS.get(name, "gray"), name)
 
    # arrows, and the points along them that site labels should keep clear of
    tips = {f: loadings[i, :2] * ARROW_SCALE for i, f in enumerate(FEATURES)}
    avoid_x, avoid_y = list(pc["PC1"]), list(pc["PC2"])
    for tip in tips.values():
        ax.arrow(0, 0, *tip, color="red", width=0.005, head_width=0.1, length_includes_head=True, zorder=4)
        s = np.linspace(0.15, 1, 8)
        avoid_x += list(tip[0] * s); avoid_y += list(tip[1] * s)
 
    grouped = {f for fs in ARROW_LABEL_GROUPS.values() for f in fs}
    arrow_texts = [place_arrow_label(ax, tip, ARROW_LABELS.get(f, f)) for f, tip in tips.items() if f not in grouped]
    arrow_texts += [place_arrow_label(ax, np.mean([tips[f] for f in fs], axis=0), lab)
                    for lab, fs in ARROW_LABEL_GROUPS.items()]
 
    # site labels: one per location, unless LABEL_ALL_POINTS
    texts = []
    if LABEL_ALL_POINTS:
        multi = df["Asukoht"].duplicated(keep=False)
        for idx, row in pc.iterrows():
            label = df.at[idx, "Asukoht"]
            if multi[idx]:
                label += f" ({df.at[idx, 'month']:02d}/{df.at[idx, 'year']})"
            texts.append(ax.text(row["PC1"], row["PC2"], label, fontsize=8))
    else:
        first = pc.groupby(df["Asukoht"]).first()
        for loc, row in first.iterrows():
            texts.append(ax.text(row["PC1"], row["PC2"], loc, fontsize=8))
 
    ax.set_xlabel(f"PC1 - {var[0]:.1f}%", fontsize=12)
    ax.set_ylabel(f"PC2 - {var[1]:.1f}%", fontsize=12)
    if title:
        ax.set_title(title, fontsize=12)
    ax.legend(fontsize=12)
    ax.grid(True)
    fig.tight_layout()
 
    fig.canvas.draw()                                   # needed to measure the arrow labels
    for t in arrow_texts:
        bx, by = text_outline_points(fig, ax, t)
        avoid_x += list(bx); avoid_y += list(by)
    spread_labels(texts, np.array(avoid_x), np.array(avoid_y))
 
    fig.savefig(f"{fname}.png", dpi=300)
    fig.savefig(f"{fname}.svg")
    plt.show()

## 6A. Biplot, original method

In [ ]:
biplot(scores_A, load_A, var_A, f"pca_biplot_original_{DATE_TAG}")

## 6B. Biplot, signed log transform on the flux columns

In [ ]:
biplot(scores_B, load_B, var_B, f"pca_biplot_transformed_{DATE_TAG}")

## 7. Comparison
**Loadings** show how strongly each variable points along PC1 and PC2. Look at whether the flux variables gain or lose weight after the transform, and whether the stratospheric variables still dominate PC1.

**Explained variance** and the scree plot show whether the transform changes how much structure the first two components capture.

In [ ]:
loadings = pd.DataFrame({
    "PC1 A": load_A[:, 0], "PC1 B": load_B[:, 0],
    "PC2 A": load_A[:, 1], "PC2 B": load_B[:, 1],
}, index=FEATURES).round(3)
loadings

In [ ]:
n = len(var_A)
x = np.arange(1, n + 1)
fig, ax = plt.subplots(figsize=(8, 6))
ax.bar(x - 0.2, var_A, width=0.4, color="green", alpha=0.7, edgecolor="black", label="A: original")
ax.bar(x + 0.2, var_B, width=0.4, color="purple", alpha=0.7, edgecolor="black", label="B: transformed fluxes")
ax.set_xlabel("Principal component", fontsize=12)
ax.set_ylabel("Explained variance (%)", fontsize=12)
ax.set_xticks(x)
ax.legend(fontsize=11)
fig.tight_layout()
fig.savefig(f"scree_plot_{DATE_TAG}.png", dpi=300)
plt.show()

pd.DataFrame({"A: original": var_A, "B: transformed": var_B}, index=[f"PC{i}" for i in x]).round(1)